<a href="https://colab.research.google.com/github/mooch443/dataset-fixer/blob/main/notebooks/04_model_bundle_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Compare model bundles from W&B or ZIP files

This notebook compares models exported by the YOLO26 semantic-segmentation,
YOLO26 instance-segmentation, and nnU-Net training notebooks. Each source can
be a W&B run, a downloaded reproducibility ZIP, or a standalone Ultralytics
`.pt`. Bundle ZIPs provide all model metadata. Standalone `.pt` files expose
their task and input resolution, while the source mapping supplies whichever
native-tile/upscale value cannot be proven from the checkpoint.

The evaluation dataset must be a fixed on-disk semantic-mask dataset accepted
by `Dataset.open()`. Instance predictions are projected to binary foreground,
so models with different instance class semantics should not be mixed blindly.

The dataset source can be a dataset folder or a ZIP containing one. In Colab,
a Drive ZIP is first copied to `/content` and extracted there, so evaluation
does not repeatedly read thousands of files through the Drive mount.

Before loading a checkpoint, the notebook validates training/evaluation
geometry. A pre-tiled 128 px dataset will therefore be rejected for a model
trained on native 256 px tiles. Mixed-size, untiled full-image datasets are
allowed; SAHI then uses each model bundle's own native training-tile size.

> Full-image SAHI comparison can be expensive. nnU-Net also uses its official
> mirroring TTA, so test the setup on a small cohort before starting a long run.

## 1. Detect Colab and mount Google Drive

In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive  # type: ignore
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

print(f"Runtime: {'Google Colab' if IN_COLAB else 'local Jupyter'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: Google Colab


## 2. Install comparison dependencies in Colab

Local environments are left unchanged. Install the project locally yourself,
or run this notebook from an environment where the checkout is importable.

In [2]:
if IN_COLAB:
    import subprocess

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            #"-q",
            (
                "dataset-fixer[comparison,sahi,nnunet] @ "
                "git+https://github.com/mooch443/dataset-fixer.git"
            ),
            "wandb>=0.19.10",
        ]
    )

In [3]:
import wandb
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: twalter (max-planck-institute-for-animal-behavior) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. Configuration

`MODEL_SOURCES` is ordered only for display. No model is a baseline: the
report computes every unordered paired comparison. Use a plain string for a
bundle ZIP or an unambiguous W&B run. A mapping additionally supports a short
display name, a specific `run_file`, or standalone `.pt` geometry.

Accepted W&B forms include `wandb:entity/project/run-id` and a full run URL.
The notebook first uses the run's `evaluation_bundle` summary field, then
falls back to one model ZIP or one `best.pt`.

`DATASET_GEOMETRY_OVERRIDE` is normally `None`: dataset-fixer exports preserve
their tiling history in `reports/dataset-info.json`. Use an override for an
external tiled dataset whose folder has no such metadata. Sizes are
`(height, width)`; a scalar means a square. For example:

```python
DATASET_GEOMETRY_OVERRIDE = {
    "native_tile_size": 128,
    "upscale_factor": 2,
}
```

In [4]:
MODEL_SOURCES = [
    {
        "source": "/content/drive/MyDrive/islands/islands-128-08.08.2026-merged-1class_yolo-128px-yolox.pt",
        "native_tile_size": 128,
        "upscale_factor": 2,
    },
    
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/0we3e4aq",
    # "/content/drive/MyDrive/models/downloaded-yolo-bundle.zip",
    # {
    #     "source": "/content/drive/MyDrive/models/best.pt",
    #     "name": "standalone-yolo",
    #     # task and resolution are normally inferred from the checkpoint.
    #     "native_tile_size": 128,  # enough to infer 2x when imgsz=256
    # },
    # {
    #     "source": "wandb:entity/project/run-id",
    #     "name": "short-display-name",
    #     "run_file": "exact-bundle-name.zip",  # or "best.pt"
    # },
]

DATASET_SOURCE = (
    "/content/drive/MyDrive/islands/islands-fair-base-sem.zip"
    if IN_COLAB
    else "/Users/tristan/Downloads/island-dataset/islands-fair-base-sem"
)
SPLIT = "val"

# None uses metadata/history from the dataset folder.
DATASET_GEOMETRY_OVERRIDE = None

# Evaluation policy. SAHI slice dimensions always come from each model bundle.
INFERENCE = "sahi"  # "sahi" or "native"
SAHI_OVERLAP = 0.15
WORKERS = 4
SAVE_PREDICTION_PLOTS = True
VALIDATE_CHECKPOINT_HASH = True

# None selects CUDA, then MPS, then CPU. A value explicitly forces that device.
DEVICE_OVERRIDE = None  # "cuda", "mps", or "cpu"

# None writes the content-addressed report below the local extracted dataset.
# Set a Drive destination explicitly if the final report must persist in Colab.
COMPARISON_DESTINATION = None

WORK_ROOT = Path(
    "/content/model-bundle-comparison"
    if IN_COLAB
    else "~/.cache/dataset-fixer/model-bundle-comparison"
).expanduser()

## 4. Imports and runtime device

In [5]:
from collections import Counter
from collections.abc import Mapping
import hashlib
import json
import math
import os
import re
import shutil
import stat
import tempfile
from urllib.parse import urlparse
from zipfile import ZipFile

import pandas as pd
import torch
from PIL import Image
from IPython.display import Image as NotebookImage, display

# Make a local source checkout importable without installing it into the kernel.
if not IN_COLAB:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        source_tree = candidate / "src"
        if (candidate / "pyproject.toml").is_file() and source_tree.is_dir():
            sys.path.insert(0, str(source_tree))
            break

from dataset_fixer import Dataset, Model


def select_device(override=None):
    if override is not None:
        selected = str(override).lower()
        if selected not in {"cuda", "mps", "cpu"}:
            raise ValueError("DEVICE_OVERRIDE must be 'cuda', 'mps', 'cpu', or None")
        if selected == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested but torch.cuda.is_available() is false")
        if selected == "mps" and not torch.backends.mps.is_available():
            raise RuntimeError("MPS was requested but torch.backends.mps.is_available() is false")
        return selected
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = select_device(DEVICE_OVERRIDE)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Inference device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: neither CUDA nor MPS is available; inference may be very slow.")

Inference device: cuda


## 5. Bundle and geometry helpers

Extraction is cached by ZIP SHA-256. Archives with absolute paths, parent
traversal, or symlinks are rejected. Checkpoint hashes are verified against
the bundle manifest by default.

In [6]:
SUPPORTED_FORMATS = {
    "dataset-fixer-nnunet-model-folder-v1",
    "ultralytics-yolo26-sem-reproducibility-bundle-v1",
    "ultralytics-yolo26-instance-seg-reproducibility-bundle-v1",
}
IMAGE_SUFFIXES = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def first_value(*values):
    return next((value for value in values if value is not None), None)


def normalize_size(value, label):
    if value is None:
        return None
    if isinstance(value, bool):
        raise ValueError(f"{label} must be a positive size, not {value!r}")
    if isinstance(value, (int, float)):
        result = (int(value), int(value))
    elif isinstance(value, (list, tuple)) and len(value) == 2:
        result = (int(value[0]), int(value[1]))
    else:
        raise ValueError(f"{label} must be an integer or two-item size, got {value!r}")
    if min(result) <= 0:
        raise ValueError(f"{label} must be positive, got {result}")
    return result


def scaled_size(size, factor):
    return tuple(int(value) * int(factor) for value in size)


def size_text(size):
    return "unknown" if size is None else f"{size[0]}x{size[1]}"


def geometry_text(geometry):
    native = size_text(geometry.get("native_tile_size"))
    factor = geometry.get("upscale_factor")
    output = size_text(geometry.get("adapter_output_size"))
    scale = "?x" if factor is None else f"{factor}x"
    return f"native {native}, {scale} -> {output}"


def manifest_geometry(manifest):
    dataset = dict(manifest.get("dataset") or {})
    model = dict(manifest.get("model") or {})
    compare = dict(manifest.get("compare_models") or {})
    train_args = dict(
        manifest.get("train_args")
        or (manifest.get("training") or {}).get("train_args")
        or {}
    )
    native = normalize_size(
        first_value(
            manifest.get("native_tile_size"),
            dataset.get("native_tile_size"),
        ),
        "bundle native_tile_size",
    )
    output = normalize_size(
        first_value(
            manifest.get("adapter_output_size"),
            dataset.get("adapter_output_size"),
            model.get("imgsz"),
            train_args.get("imgsz"),
        ),
        "bundle adapter_output_size",
    )
    factor = first_value(
        compare.get("upscale_factor"),
        manifest.get("upscale_factor"),
        dataset.get("upscale_factor"),
    )
    if factor is None and native is not None and output is not None:
        ratios = (output[0] / native[0], output[1] / native[1])
        if ratios[0] == ratios[1] and ratios[0].is_integer():
            factor = int(ratios[0])
    if factor is not None:
        if isinstance(factor, bool) or int(factor) <= 0 or int(factor) != factor:
            raise ValueError(f"Bundle upscale_factor must be a positive integer, got {factor!r}")
        factor = int(factor)
    if output is None and native is not None and factor is not None:
        output = scaled_size(native, factor)
    geometry = {
        "native_tile_size": native,
        "upscale_factor": factor,
        "adapter_output_size": output,
    }
    missing = [key for key, value in geometry.items() if value is None]
    if missing:
        raise ValueError(
            "Bundle does not contain complete model geometry; missing "
            + ", ".join(missing)
        )
    expected = scaled_size(native, factor)
    if output != expected:
        raise ValueError(
            f"Bundle geometry is internally inconsistent: {geometry_text(geometry)}; "
            f"native size times upscale factor is {size_text(expected)}"
        )
    return geometry


def normalize_wandb_run(source):
    value = str(source).strip()
    if value.startswith("wandb:"):
        value = value.removeprefix("wandb:").strip("/")
    if "wandb.ai/" in value:
        parsed = urlparse(value if "://" in value else f"https://{value}")
        parts = [part for part in parsed.path.split("/") if part]
    else:
        parts = [part for part in value.split("/") if part]
    if len(parts) == 4 and parts[2] == "runs":
        parts = parts[:2] + parts[3:]
    if len(parts) != 3:
        raise ValueError(
            "W&B source must be wandb:entity/project/run-id or a full run URL, "
            f"got {source!r}"
        )
    return "/".join(parts)


def choose_run_file(run, requested=None):
    files = {file.name: file for file in run.files()}

    def matches(name):
        return [
            candidate
            for candidate in files
            if candidate == name or Path(candidate).name == Path(name).name
        ]

    preferred = requested or dict(run.summary).get("evaluation_bundle")
    if preferred:
        selected = matches(str(preferred))
        if len(selected) == 1:
            return files[selected[0]]
        if requested:
            raise FileNotFoundError(
                f"Requested W&B bundle {requested!r} is not an unambiguous run file"
            )
    zip_candidates = sorted(name for name in files if name.lower().endswith(".zip"))
    if len(zip_candidates) == 1:
        return files[zip_candidates[0]]
    checkpoint_candidates = sorted(
        name for name in files if Path(name).name.lower() == "best.pt"
    )
    if not zip_candidates and len(checkpoint_candidates) == 1:
        return files[checkpoint_candidates[0]]
    candidates = zip_candidates + checkpoint_candidates
    rendered = "\n".join(f"  - {name}" for name in candidates) or "  (none)"
    raise RuntimeError(
        "Could not identify one model bundle ZIP or best.pt in the W&B run. "
        f"Set run_file in MODEL_SOURCES. Candidates:\n{rendered}"
    )


def download_wandb_model(source, run_file=None):
    import wandb

    run_path = normalize_wandb_run(source)
    run = wandb.Api().run(run_path)
    remote = choose_run_file(run, requested=run_file)
    destination = WORK_ROOT / "downloads" / "wandb" / run.entity / run.project / run.id
    destination.mkdir(parents=True, exist_ok=True)
    downloaded = remote.download(root=str(destination), exist_ok=True)
    try:
        path = Path(downloaded.name).expanduser().resolve()
    finally:
        downloaded.close()
    if path.suffix.lower() not in {".zip", ".pt"}:
        raise ValueError(f"W&B model source is neither a ZIP nor .pt: {path}")
    return path, run


def safe_extract(archive_path, destination):
    root = destination.resolve()
    with ZipFile(archive_path) as archive:
        for info in archive.infolist():
            member = Path(info.filename)
            mode = info.external_attr >> 16
            if member.is_absolute() or ".." in member.parts:
                raise ValueError(f"Unsafe ZIP member path: {info.filename!r}")
            if stat.S_ISLNK(mode):
                raise ValueError(f"ZIP symlinks are not accepted: {info.filename!r}")
            target = (root / member).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f"ZIP member escapes extraction root: {info.filename!r}")
        archive.extractall(root)


def one_path(paths, label):
    values = sorted(set(Path(path) for path in paths))
    if len(values) != 1:
        rendered = "\n".join(f"  - {path}" for path in values) or "  (none)"
        raise RuntimeError(f"Expected exactly one {label}; found {len(values)}:\n{rendered}")
    return values[0]


def extract_bundle(zip_path):
    zip_path = Path(zip_path).expanduser().resolve()
    if not zip_path.is_file() or zip_path.suffix.lower() != ".zip":
        raise FileNotFoundError(f"Model source is not a ZIP file: {zip_path}")
    archive_hash = sha256_file(zip_path)
    extraction_root = WORK_ROOT / "extracted" / archive_hash[:20]
    manifests = list(extraction_root.rglob("bundle_manifest.json")) if extraction_root.exists() else []
    if len(manifests) == 1:
        return extraction_root, manifests[0], archive_hash
    if extraction_root.exists():
        raise RuntimeError(
            f"Cached extraction is incomplete or ambiguous: {extraction_root}. "
            "Remove that one cache directory and retry."
        )
    extraction_root.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(
        tempfile.mkdtemp(prefix=f".{archive_hash[:12]}-", dir=extraction_root.parent)
    )
    try:
        safe_extract(zip_path, temporary)
        one_path(temporary.rglob("bundle_manifest.json"), "bundle_manifest.json")
        temporary.replace(extraction_root)
    except BaseException:
        shutil.rmtree(temporary, ignore_errors=True)
        raise
    manifest_path = one_path(
        extraction_root.rglob("bundle_manifest.json"), "bundle_manifest.json"
    )
    return extraction_root, manifest_path, archive_hash


def short_model_name(raw_name, identity):
    value = re.sub(r"\s+", " ", str(raw_name)).strip() or "model"
    suffix = hashlib.sha256(str(identity).encode()).hexdigest()[:8]
    if len(value.encode("utf-8")) > 96:
        value = f"{value[:72].rstrip()}-{suffix}"
    return value


def normalize_yolo_task(value):
    raw = str(value or "").lower().replace("_", "-")
    if raw in {"semantic", "semantic-segmentation", "semantic-segment"}:
        return "semantic"
    if raw in {"segment", "instance-segmentation"}:
        return "segment"
    raise ValueError(f"Unsupported or missing Ultralytics task: {value!r}")


def resolve_standalone_checkpoint(path, spec, source, run=None):
    """Infer safe checkpoint metadata and require unprovable tile geometry."""

    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    try:
        if not isinstance(checkpoint, Mapping):
            raise ValueError(f"Unsupported Ultralytics checkpoint structure: {type(checkpoint)}")
        train_args = dict(checkpoint.get("train_args") or {})
        serialized_model = checkpoint.get("model") or checkpoint.get("ema")
        model_args = dict(getattr(serialized_model, "args", {}) or {})
    finally:
        del checkpoint

    run_config = dict(run.config) if run is not None else {}
    reproducibility = dict(run_config.get("reproducibility") or {})
    run_dataset = dict(run_config.get("dataset") or {})
    task = normalize_yolo_task(
        first_value(
            spec.get("task"),
            train_args.get("task"),
            model_args.get("task"),
            run_config.get("task"),
        )
    )
    output = normalize_size(
        first_value(
            spec.get("resolution"),
            spec.get("adapter_output_size"),
            train_args.get("imgsz"),
            model_args.get("imgsz"),
            reproducibility.get("adapter_output_size"),
            run_config.get("adapter_output_size"),
            run_dataset.get("adapter_output_size"),
        ),
        "standalone checkpoint resolution",
    )
    native = normalize_size(
        first_value(
            spec.get("native_tile_size"),
            reproducibility.get("native_tile_size"),
            run_config.get("native_tile_size"),
            run_dataset.get("native_tile_size"),
        ),
        "standalone checkpoint native_tile_size",
    )
    factor = first_value(
        spec.get("upscale_factor"),
        reproducibility.get("upscale_factor"),
        run_config.get("upscale_factor"),
        run_dataset.get("upscale_factor"),
    )
    if factor is not None:
        if isinstance(factor, bool) or int(factor) <= 0 or int(factor) != factor:
            raise ValueError("Standalone checkpoint upscale_factor must be a positive integer")
        factor = int(factor)
    if output is None:
        raise ValueError(f"Standalone checkpoint has no usable imgsz/resolution metadata: {path}")
    if native is None and factor is None:
        raise ValueError(
            f"Standalone checkpoint {path} cannot prove its native training-tile size. "
            "Set native_tile_size or upscale_factor in its MODEL_SOURCES mapping."
        )
    if native is None:
        if output[0] % factor or output[1] % factor:
            raise ValueError("Checkpoint resolution is not divisible by the supplied upscale_factor")
        native = (output[0] // factor, output[1] // factor)
    if factor is None:
        ratios = (output[0] / native[0], output[1] / native[1])
        if ratios[0] != ratios[1] or not ratios[0].is_integer():
            raise ValueError("Checkpoint native tile and resolution do not define one integer upscale")
        factor = int(ratios[0])
    geometry = {
        "native_tile_size": native,
        "upscale_factor": factor,
        "adapter_output_size": output,
    }
    if output != scaled_size(native, factor):
        raise ValueError(f"Standalone checkpoint geometry is inconsistent: {geometry_text(geometry)}")
    if output[0] != output[1]:
        raise ValueError("dataset-fixer currently requires a scalar Ultralytics resolution")

    actual_hash = sha256_file(path)
    summary = dict(run.summary) if run is not None else {}
    expected_hash = first_value(
        spec.get("checkpoint_sha256"),
        summary.get("selected_checkpoint_sha256"),
        (run_config.get("training_outcome") or {}).get("checkpoint_sha256"),
    )
    if VALIDATE_CHECKPOINT_HASH and expected_hash and actual_hash != str(expected_hash):
        raise ValueError(
            f"Checkpoint SHA-256 mismatch for {path}: expected {expected_hash}, got {actual_hash}"
        )
    inferred_name = first_value(
        spec.get("name"),
        getattr(run, "display_name", None) if run is not None else None,
        path.stem,
    )
    return {
        "name": short_model_name(inferred_name, actual_hash),
        "source": source,
        "zip_path": None,
        "archive_sha256": None,
        "manifest_path": None,
        "manifest": {},
        "format": "standalone-ultralytics-checkpoint",
        "kind": "ultralytics",
        "task": task,
        "model_path": path,
        "checkpoint_path": path,
        "checkpoint": path.name,
        "checkpoint_sha256": actual_hash,
        "trainer": None,
        "folds": (),
        "resolution": output[0],
        "geometry": geometry,
    }


def resolve_model_source(source_spec):
    if isinstance(source_spec, str):
        spec = {"source": source_spec}
    elif isinstance(source_spec, Mapping):
        spec = dict(source_spec)
    else:
        raise TypeError("Each MODEL_SOURCES entry must be a string or mapping")
    source = str(spec.get("source") or "").strip()
    if not source:
        raise ValueError("Each MODEL_SOURCES mapping requires a non-empty source")

    run = None
    if source.startswith("wandb:") or "wandb.ai/" in source:
        model_source, run = download_wandb_model(
            source,
            first_value(spec.get("run_file"), spec.get("bundle_file")),
        )
    else:
        model_source = Path(source).expanduser().resolve()

    if model_source.suffix.lower() == ".pt":
        if not model_source.is_file():
            raise FileNotFoundError(model_source)
        return resolve_standalone_checkpoint(model_source, spec, source, run=run)
    if model_source.suffix.lower() != ".zip":
        raise ValueError(f"Model source must be a bundle ZIP, .pt, or W&B run: {model_source}")

    zip_path = model_source
    extraction_root, manifest_path, archive_hash = extract_bundle(zip_path)
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    bundle_format = str(manifest.get("format") or "")
    if bundle_format not in SUPPORTED_FORMATS:
        raise ValueError(
            f"Unsupported bundle format {bundle_format!r} in {manifest_path}; "
            f"supported formats are {sorted(SUPPORTED_FORMATS)}"
        )
    geometry = manifest_geometry(manifest)
    manifest_root = manifest_path.parent
    model_metadata = dict(manifest.get("model") or {})
    compare_metadata = dict(manifest.get("compare_models") or {})

    if bundle_format == "dataset-fixer-nnunet-model-folder-v1":
        kind = "nnunet"
        task = "semantic"
        trainer = str(
            first_value(manifest.get("trainer"), model_metadata.get("trainer")) or ""
        )
        if not trainer:
            raise ValueError("nnU-Net bundle lacks trainer metadata")
        if not (manifest_root / "dataset.json").is_file() or not (manifest_root / "plans.json").is_file():
            model_root = one_path(
                [
                    path
                    for path in extraction_root.rglob("plans.json")
                    if (path.parent / "dataset.json").is_file()
                ],
                "nnU-Net model folder",
            ).parent
        else:
            model_root = manifest_root
        checkpoint = str(
            first_value(
                compare_metadata.get("checkpoint"),
                manifest.get("checkpoint"),
                model_metadata.get("checkpoint"),
            )
            or ""
        )
        folds_value = first_value(
            compare_metadata.get("folds"),
            manifest.get("folds"),
            [model_metadata.get("fold")] if model_metadata.get("fold") is not None else None,
        )
        folds = tuple(str(value) for value in (folds_value or ()))
        if not checkpoint or not folds:
            raise ValueError("nnU-Net bundle lacks checkpoint or folds metadata")
        checkpoint_path = one_path(
            [model_root / f"fold_{fold}" / checkpoint for fold in folds],
            "nnU-Net checkpoint",
        )
        resolution = None
    else:
        kind = "ultralytics"
        trainer = None
        task = normalize_yolo_task(model_metadata.get("task"))
        checkpoint = str(model_metadata.get("checkpoint") or "best.pt")
        checkpoint_path = one_path(
            manifest_root.rglob(Path(checkpoint).name), "YOLO best checkpoint"
        )
        model_root = checkpoint_path
        folds = ()
        output_size = geometry["adapter_output_size"]
        if output_size[0] != output_size[1]:
            raise ValueError(
                "dataset-fixer currently requires a scalar Ultralytics resolution; "
                f"bundle declares {output_size}"
            )
        resolution = output_size[0]

    expected_hash = first_value(
        model_metadata.get("checkpoint_sha256"),
        manifest.get("checkpoint_sha256"),
    )
    actual_hash = sha256_file(checkpoint_path) if VALIDATE_CHECKPOINT_HASH else None
    if expected_hash and actual_hash != str(expected_hash):
        raise ValueError(
            f"Checkpoint SHA-256 mismatch for {checkpoint_path}: "
            f"expected {expected_hash}, got {actual_hash}"
        )

    run_metadata = dict(manifest.get("run") or {})
    inferred_name = first_value(
        spec.get("name"),
        run_metadata.get("wandb_run_name"),
        getattr(run, "display_name", None) if run is not None else None,
        zip_path.stem,
    )
    name = short_model_name(inferred_name, actual_hash or archive_hash)
    return {
        "name": name,
        "source": source,
        "zip_path": zip_path,
        "archive_sha256": archive_hash,
        "manifest_path": manifest_path,
        "manifest": manifest,
        "format": bundle_format,
        "kind": kind,
        "task": task,
        "model_path": model_root,
        "checkpoint_path": checkpoint_path,
        "checkpoint": checkpoint,
        "checkpoint_sha256": actual_hash or expected_hash,
        "trainer": trainer,
        "folds": folds,
        "resolution": resolution,
        "geometry": geometry,
    }

## 6. Open the evaluation dataset and establish its geometry

The guard reads the most recent `tile-*` operation from dataset-fixer history.
It also checks actual image dimensions. For an external tiled dataset, set
`DATASET_GEOMETRY_OVERRIDE`; an override wins over inferred metadata.

In [7]:
def find_dataset_root(extraction_root, split):
    candidates = []
    for path in (extraction_root, *extraction_root.rglob("*")):
        if not path.is_dir() or "__MACOSX" in path.parts:
            continue
        if (path / split / "images").is_dir() and (path / split / "masks").is_dir():
            candidates.append(path)
    return one_path(candidates, f"semantic dataset root containing {split}/images and {split}/masks")


def resolve_dataset_source(source, split):
    source_path = Path(source).expanduser().resolve()
    if source_path.is_dir():
        return source_path
    if not source_path.is_file() or source_path.suffix.lower() != ".zip":
        raise FileNotFoundError(
            f"DATASET_SOURCE must be a dataset directory or ZIP file: {source_path}"
        )

    archive_path = source_path
    if IN_COLAB:
        # The size suffix avoids silently reusing a same-named Drive archive.
        local_archives = WORK_ROOT / "datasets" / "archives"
        local_archives.mkdir(parents=True, exist_ok=True)
        archive_path = local_archives / (
            f"{source_path.stem}-{source_path.stat().st_size}{source_path.suffix.lower()}"
        )
        if not archive_path.is_file():
            temporary_copy = archive_path.with_suffix(archive_path.suffix + ".copying")
            shutil.copy2(source_path, temporary_copy)
            temporary_copy.replace(archive_path)
        print(f"Local dataset ZIP: {archive_path}")

    archive_hash = sha256_file(archive_path)
    extraction_root = WORK_ROOT / "datasets" / "extracted" / archive_hash[:20]
    if extraction_root.exists():
        dataset_root = find_dataset_root(extraction_root, split)
        print(f"Reusing extracted dataset: {dataset_root}")
        return dataset_root

    extraction_root.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(
        tempfile.mkdtemp(prefix=f".{archive_hash[:12]}-", dir=extraction_root.parent)
    )
    try:
        safe_extract(archive_path, temporary)
        find_dataset_root(temporary, split)
        temporary.replace(extraction_root)
    except BaseException:
        shutil.rmtree(temporary, ignore_errors=True)
        raise
    dataset_root = find_dataset_root(extraction_root, split)
    print(f"Extracted dataset locally: {dataset_root}")
    return dataset_root


def inspect_dataset_geometry(dataset, split, override=None):
    if split not in dataset.splits:
        raise ValueError(f"Unknown split {split!r}; available splits: {dataset.splits}")
    manifest = dict(dataset.manifest or {})
    history = list(manifest.get("history") or [])
    native = None
    source = "mixed-size/untiled dataset"
    for event in reversed(history):
        operation = str(event.get("operation") or "")
        settings = dict(event.get("settings") or {})
        if operation == "tile" or operation.startswith("tile-"):
            native = normalize_size(settings.get("tile_size"), "dataset tile_size")
            source = f"dataset history operation {operation!r}"
            break

    factor = None
    output = None
    if override is not None:
        if not isinstance(override, Mapping):
            raise TypeError("DATASET_GEOMETRY_OVERRIDE must be a mapping or None")
        native = normalize_size(
            first_value(override.get("native_tile_size"), native),
            "dataset native_tile_size",
        )
        factor_value = override.get("upscale_factor")
        if factor_value is not None:
            if isinstance(factor_value, bool) or int(factor_value) <= 0 or int(factor_value) != factor_value:
                raise ValueError("Dataset upscale_factor must be a positive integer")
            factor = int(factor_value)
        output = normalize_size(
            override.get("adapter_output_size"), "dataset adapter_output_size"
        )
        source = "DATASET_GEOMETRY_OVERRIDE"

    if factor is not None and native is None:
        raise ValueError("Dataset upscale_factor requires native_tile_size")
    if output is None and native is not None and factor is not None:
        output = scaled_size(native, factor)
    if factor is None and native is not None and output is not None:
        ratios = (output[0] / native[0], output[1] / native[1])
        if ratios[0] != ratios[1] or not ratios[0].is_integer():
            raise ValueError("Dataset native/output sizes do not define one integer upscale factor")
        factor = int(ratios[0])
    if native is not None and factor is not None and output != scaled_size(native, factor):
        raise ValueError(
            "Dataset geometry override is internally inconsistent: "
            f"native {native}, factor {factor}, output {output}"
        )

    image_root = Path(dataset.image_dirs[split])
    image_paths = sorted(
        path for path in image_root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    if not image_paths:
        raise ValueError(f"Dataset split contains no supported images: {image_root}")
    dimensions = Counter()
    for path in image_paths:
        with Image.open(path) as image:
            dimensions[(image.height, image.width)] += 1
    uniform_size = next(iter(dimensions)) if len(dimensions) == 1 else None

    if native is not None:
        accepted_actual_sizes = {native}
        if output is not None:
            accepted_actual_sizes.add(output)
        if uniform_size is None:
            raise ValueError(
                f"Dataset declares tiled geometry {size_text(native)} via {source}, "
                f"but split {split!r} has {len(dimensions)} image dimensions"
            )
        if uniform_size not in accepted_actual_sizes:
            raise ValueError(
                f"Dataset declares {geometry_text({'native_tile_size': native, 'upscale_factor': factor, 'adapter_output_size': output})} "
                f"via {source}, but actual {split!r} images are {size_text(uniform_size)}"
            )

    return {
        "native_tile_size": native,
        "upscale_factor": factor,
        "adapter_output_size": output,
        "source": source,
        "images": len(image_paths),
        "dimension_counts": dimensions,
        "uniform_image_size": uniform_size,
    }


DATASET_FOLDER = resolve_dataset_source(DATASET_SOURCE, SPLIT)
dataset = Dataset.open(DATASET_FOLDER)
if dataset.format != "semantic_masks":
    raise ValueError(
        f"This notebook requires a semantic-mask dataset, got format={dataset.format!r}"
    )
DATASET_GEOMETRY = inspect_dataset_geometry(
    dataset,
    SPLIT,
    override=DATASET_GEOMETRY_OVERRIDE,
)

print(f"Dataset: {dataset.location}")
print(f"Split: {SPLIT} ({DATASET_GEOMETRY['images']} images)")
if DATASET_GEOMETRY["native_tile_size"] is None:
    print("Geometry: mixed-size/untiled; each bundle supplies its own SAHI tile geometry")
else:
    print(
        f"Geometry: {geometry_text(DATASET_GEOMETRY)} "
        f"({DATASET_GEOMETRY['source']})"
    )

Local dataset ZIP: /content/model-bundle-comparison/datasets/archives/islands-fair-base-sem-2598934931.zip
Reusing extracted dataset: /content/model-bundle-comparison/datasets/extracted/176193dbf12e2f06f942


Loading semantic-mask dataset:   0%|          | 0/4692 [00:00<?, ?pair/s]

Validating dataset:   0%|          | 0/4692 [00:00<?, ?image/s]

Dataset: /content/model-bundle-comparison/datasets/extracted/176193dbf12e2f06f942
Split: val (1205 images)
Geometry: mixed-size/untiled; each bundle supplies its own SAHI tile geometry


## 7. Download, extract, validate, and load models

No inference starts in this section. All bundle hashes, internal geometry,
dataset/model geometry, tasks, and model paths are checked first.

In [8]:
def geometry_mismatches(dataset_geometry, model_geometry):
    mismatches = []
    for key, label in (
        ("native_tile_size", "native tile size"),
        ("upscale_factor", "upscale factor"),
        ("adapter_output_size", "adapter output size"),
    ):
        dataset_value = dataset_geometry.get(key)
        model_value = model_geometry.get(key)
        if dataset_value is not None and dataset_value != model_value:
            mismatches.append(f"{label}: dataset={dataset_value}, model={model_value}")
    return mismatches


def ensure_nnunet_trainer_available(trainer_name):
    """Restore training-length-only trainer variants in a fresh runtime.

    Official nnU-Net needs the original trainer class to reconstruct the
    architecture, even when that subclass changed only ``num_epochs``. The
    producer bundles record the class name. Standard classes resolve directly;
    a custom ``nnUNetTrainer_<N>epochs`` gets an equivalent external trainer in
    WORK_ROOT rather than modifying site-packages.
    """
    from nnunetv2.utilities.find_objects import recursive_find_trainer_class_by_name

    try:
        return recursive_find_trainer_class_by_name(trainer_name)
    except RuntimeError as original_error:
        match = re.fullmatch(r"nnUNetTrainer_(\d+)epochs", str(trainer_name))
        if match is None:
            raise RuntimeError(
                f"nnU-Net trainer {trainer_name!r} is unavailable and is not a "
                "supported training-length-only variant"
            ) from original_error

    epochs = int(match.group(1))
    trainer_root = WORK_ROOT / "nnunet_external_trainers"
    trainer_root.mkdir(parents=True, exist_ok=True)
    trainer_file = trainer_root / f"{trainer_name}.py"
    source = f'''\
import torch

from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


class {trainer_name}(nnUNetTrainer):
    def __init__(self, plans, configuration, fold, dataset_json,
                 device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = {epochs}
'''
    if not trainer_file.is_file() or trainer_file.read_text(encoding="utf-8") != source:
        trainer_file.write_text(source, encoding="utf-8")
    existing = [value for value in os.environ.get("nnUNet_extTrainer", "").split(os.pathsep) if value]
    if str(trainer_root) not in existing:
        os.environ["nnUNet_extTrainer"] = os.pathsep.join([str(trainer_root), *existing])
    resolved = recursive_find_trainer_class_by_name(trainer_name)
    print(f"Restored external nnU-Net trainer: {trainer_name} ({epochs} epochs)")
    return resolved


if not MODEL_SOURCES:
    raise ValueError("Add at least one W&B run, bundle ZIP, or .pt to MODEL_SOURCES")
if INFERENCE not in {"sahi", "native"}:
    raise ValueError("INFERENCE must be 'sahi' or 'native'")
if not math.isfinite(float(SAHI_OVERLAP)) or not 0 <= float(SAHI_OVERLAP) < 1:
    raise ValueError("SAHI_OVERLAP must be finite and in [0, 1)")

if any(
    str(value if isinstance(value, str) else value.get("source", "")).startswith("wandb:")
    or "wandb.ai/" in str(value if isinstance(value, str) else value.get("source", ""))
    for value in MODEL_SOURCES
):
    import wandb

    wandb.login()

bundles = [resolve_model_source(source) for source in MODEL_SOURCES]
for bundle in bundles:
    if bundle["kind"] == "nnunet":
        ensure_nnunet_trainer_available(bundle["trainer"])
geometry_errors = []
for bundle in bundles:
    mismatches = geometry_mismatches(DATASET_GEOMETRY, bundle["geometry"])
    if mismatches:
        geometry_errors.append(
            f"- {bundle['name']}: dataset {geometry_text(DATASET_GEOMETRY)}; "
            f"model {geometry_text(bundle['geometry'])}; "
            + "; ".join(mismatches)
        )
if geometry_errors:
    raise ValueError(
        "Dataset/model geometry mismatch; inference was not started.\n"
        + "\n".join(geometry_errors)
        + "\nUse a matching tiled evaluation dataset, a full-resolution untiled dataset, "
        "or correct DATASET_GEOMETRY_OVERRIDE if external metadata is missing."
    )

model_configurations = {}
used_names = set()
for bundle in bundles:
    name = bundle["name"]
    if name in used_names:
        name = f"{name}-{str(bundle['checkpoint_sha256'])[:8]}"
    if name in used_names:
        raise ValueError(f"Could not derive a unique model name for {bundle['source']}")
    used_names.add(name)
    bundle["name"] = name

    geometry = bundle["geometry"]
    configuration = {
        "path": str(bundle["model_path"]),
        "task": bundle["task"],
        "inference": INFERENCE,
        "device": DEVICE,
    }
    if bundle["kind"] == "nnunet":
        configuration.update(
            {
                "folds": bundle["folds"],
                "checkpoint": bundle["checkpoint"],
                "upscale_factor": geometry["upscale_factor"],
                "workers": int(WORKERS),
            }
        )
    else:
        configuration["resolution"] = bundle["resolution"]
    if INFERENCE == "sahi":
        native_height, native_width = geometry["native_tile_size"]
        configuration.update(
            {
                "sahi_slice_height": native_height,
                "sahi_slice_width": native_width,
                "sahi_overlap": float(SAHI_OVERLAP),
            }
        )
    model_configurations[name] = configuration

models = Model.load_many(model_configurations)

rows = []
for bundle in bundles:
    configuration = model_configurations[bundle["name"]]
    rows.append(
        {
            "name": bundle["name"],
            "kind": bundle["kind"],
            "task": bundle["task"],
            "device": DEVICE,
            "inference": INFERENCE,
            "native_tile": size_text(bundle["geometry"]["native_tile_size"]),
            "upscale": bundle["geometry"]["upscale_factor"],
            "model_input": size_text(bundle["geometry"]["adapter_output_size"]),
            "resolution": bundle["resolution"],
            "checkpoint": bundle["checkpoint"],
            "trainer": bundle["trainer"],
            "folds": ",".join(bundle["folds"]),
            "source": str(bundle["zip_path"] or bundle["model_path"]),
        }
    )

display(pd.DataFrame(rows))
print("Paired comparisons: every unordered model pair (no baseline)")

Trainer 'nnUNetTrainer_184epochs' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /content/model-bundle-comparison/nnunet_external_trainers
Searching for class nnUNetTrainer_184epochs in folder /content/model-bundle-comparison/nnunet_external_trainers with current module None
  Inspecting module: nnUNetTrainer_184epochs
Found class nnUNetTrainer_184epochs in nnUNetTrainer_184epochs
Using trainer 'nnUNetTrainer_184epochs' from: /content/model-bundle-comparison/nnunet_external_trainers
Restored external nnU-Net trainer: nnUNetTrainer_184epochs (184 epochs)


,name,kind,task,device,inference,native_tile,upscale,model_input,resolution,checkpoint,trainer,folds,source
0,islands-128-08.08.2026-merged-1class_yolo-128p...,ultralytics,segment,cuda,sahi,128x128,2,256x256,256.0,islands-128-08.08.2026-merged-1class_yolo-128p...,None,,/content/drive/MyDrive/islands/islands-128-08....
1,island-tiles-128-nnUNetResEncM-official-heldou...,nnunet,semantic,cuda,sahi,128x128,4,512x512,NaN,checkpoint_best_mean_fg_dice.pth,nnUNetTrainer_184epochs,0,/content/model-bundle-comparison/downloads/wan...


Paired comparisons: every unordered model pair (no baseline)


## 8. Run the fixed-cohort comparison

Completed per-model predictions are cached by dataset-fixer. A cancelled
nnU-Net model run currently has no partial-tile resume, so avoid interrupting
a long run after the geometry table has been verified.

In [10]:
destination = (
    None
    if COMPARISON_DESTINATION is None
    else Path(COMPARISON_DESTINATION).expanduser().resolve()
)
comparison = models.compare(
    dataset,
    split=SPLIT,
    save_prediction_plots=SAVE_PREDICTION_PLOTS,
    paired_comparisons="all",
    destination=destination,
)

islands-128-08.08.2026-merged-1class_yolo-128px-yolox SAHI 0.7:   0%|          | 0/1205 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 9. Inspect the result

In [ ]:
ranking = pd.DataFrame(comparison.ranking)
preferred_columns = [
    "rank",
    "model",
    "score",
    "dice",
    "iou",
    "micro_dice",
    "micro_iou",
    "inference_seconds",
    "throughput_cases_per_second",
    "cache",
]
display(ranking[[column for column in preferred_columns if column in ranking.columns]])

result_root = Path(comparison.location)
print(f"Report: {result_root}")
for filename in ("plots.png", "comparison.png"):
    path = result_root / "reports" / filename
    if path.is_file():
        display(NotebookImage(filename=str(path)))